In [ ]:
'''
Notebook author: Thuy Duong Ngo
Some implementation details are inspired by code in this repo: <https://github.com/tyui592/A_Learned_Representation_For_Artistic_Style>.
'''

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# force_remount = True

In [ ]:
# Import cell:
import cv2
import os
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
from torchvision import models
import torchvision.transforms as transforms
from torchvision.transforms.functional import to_pil_image
from torch.utils.data import Dataset
import torchvision.transforms as T
from PIL import Image
from torchvision import datasets
from torch.utils.data import DataLoader
from pathlib import Path

import random
import torch
import argparse
from pathlib import Path
from torch.optim import Adam
from torchvision.models import vgg16, VGG16_Weights
from torchvision.models.feature_extraction import create_feature_extractor

In [ ]:
# Global variables:
NUM_STYLE = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 16
lr = 1e-3
content_weight = 1

In [ ]:
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
normalize = T.Normalize(mean=MEAN, std=STD)
denormalize = T.Normalize(mean=[-m/s for m, s in zip(MEAN, STD)],
                          std=[1/std for std in STD])

def get_transforms(imsize=None, cropsize=None, cencrop=False):
    """This function transforms the image"""
    transformer = []
    if imsize:
        transformer.append(T.Resize(imsize))
    if cropsize:
        if cencrop:
            transformer.append(T.CenterCrop(cropsize))
        else:
            transformer.append(T.RandomCrop(cropsize))

    transformer.append(T.ToTensor())
    transformer.append(normalize)
    return T.Compose(transformer)


def imload(path, imsize=None, cropsize=None, cencrop=False):
    """This function loads an image"""
    transformer = get_transforms(imsize=imsize,
                                 cropsize=cropsize,
                                 cencrop=cencrop)
    image = Image.open(path).convert("RGB")
    return transformer(image).unsqueeze(0)


def imsave(image, save_path):
    """This function saves an image"""
    image = denormalize(torchvision.utils.make_grid(image)).clamp_(0.0, 1.0)
    torchvision.utils.save_image(image, save_path)
    return None

class ImageDataset:
    """Image Dataset: define an instance of the content image dataset."""

    def __init__(self, dir_path):
        self.images = sorted(list(dir_path.glob('*.jpg'))+list(dir_path.glob('*.png')))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img = Image.open(self.images[index]).convert('RGB')
        return img, index


class StyleDataset:
  """ StyleDataset: define an instance of the style dataset, in which each cinematic style is represented through
  a set of style-associated images"""
    def __init__(self, dir_path):
        self.images = []
        style_subdirs = sorted([d for d in dir_path.iterdir() if d.is_dir() and not d.name.startswith('.')])

        for style_id, style_subdir in enumerate(style_subdirs):
            self.images.extend([(img_path, style_id) for img_path in style_subdir.glob('*.png')]+[(img_path, style_id) for img_path in style_subdir.glob('*.jpg')])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path, style_id = self.images[index]
        img = Image.open(img_path).convert('RGB')
        return img, style_id


class DataProcessor:
  def __init__(self, imsize = 256, cropsize = 240, cencrop = False):
    self. transforms = get_transforms(imsize = imsize, cropsize = cropsize, cencrop = cencrop)
  def __call__(self, batch):
        """Process the batch."""
        images, indices = list(zip(*batch))

        inputs = torch.stack([self.transforms(image) for image in images])
        return inputs, indices

def resize(np_img,basewidth=200):
    '''This function resizes an image'''
    img = Image.fromarray(np_img)
    wpercent = (basewidth / float(img.size[0]))
    hsize = int((float(img.size[1]) * float(wpercent)))
    img = img.resize((basewidth, hsize), Image.LANCZOS)
    return np.array(img)

In [ ]:
'''This is customized to the links of my content dataset and my style dataset folders.
Using the DataLoader function wraps an iterable around the style and content images .
'''
# Step 1:
style_dataset_path = "/content/drive/MyDrive/Movie_Project/Directors_still_images"
content_dataset_path = "/content/drive/MyDrive/Movie_Project/Content_image/"

style_dataset = StyleDataset(dir_path = Path(style_dataset_path))
content_dataset = ImageDataset(dir_path = Path(content_dataset_path))
data_processor = DataProcessor(imsize = 256, cropsize = 240, cencrop = False)

# Step 2:
style_dataloader = torch.utils.data.DataLoader(style_dataset, batch_size = 16, shuffle = True, collate_fn = data_processor)
content_dataloader = torch.utils.data.DataLoader(content_dataset, batch_size = 16, shuffle = True, collate_fn = data_processor)

In [ ]:
def evaluate(content_path, style_index, model):
  '''This function returns the stylized image corresponding to a certain style index and a content image'''
    content_image = imload(content_path, imsize=256)
    if style_index == -1:
        style_code = torch.eye(NUM_STYLE).unsqueeze(-1)
        content_image = content_image.repeat(NUM_STYLE, 1, 1, 1)
    elif style_index in range(NUM_STYLE):
        style_code = torch.zeros(1, NUM_STYLE, 1)
        style_code[:, style_index, :] = 1
    else:
        raise RuntimeError("Not expected")
    stylized_image = model(content_image, style_code) # added clamping to simulate torch.sigmoid
    return stylized_image